# Upload a schedule to an Ambyte over USB

Finds the Ambyte plugged into this computer, pushes the schedule below down its serial
console, lets the device compile-check it, installs it and confirms it is running.
No SD card, Wi-Fi or MQTT is involved.

**Before you run it**

- Firmware v2.0.0 or later (the YAML scheduler). Older firmware is refused with a clear message.
- `pip install pyserial` in the kernel's environment.
- Close every other program holding the port (PlatformIO monitor, flash GUI, PuTTY).
- A freshly plugged or reset board needs about 30 s before its console answers.

**What it changes on the device**

- The installed schedule is *replaced as a whole*: only the jobs in the YAML below will run.
  The previous file is kept on the device as `/littlefs/schedule.yaml.bak`.
- A schedule the device compiler rejects is never installed; the running one stays untouched.
- The device restarts its schedule runner in place. It does not reboot.

Run the cells top to bottom.

In [1]:
import base64
import hashlib
import re
import sys
import time
from pathlib import Path

# The serial console helpers live in flash_gui/; find the checkout from wherever the kernel started.
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "flash_gui" / "ambyte_serial.py").is_file()), None)
if REPO is None:
    raise RuntimeError("Open this notebook from inside the ambyte-IoT checkout.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from flash_gui import ambyte_serial
from flash_gui.ambyte_serial import (FAILURE_MARK, PROMPT, SCHEDULE_PUT_CHUNK_BYTES,
                                     AmbyteConsole, ConsoleError)

print(f"repo: {REPO}")

repo: c:\Users\LudovicoCaracciolo\Documents\Git-repo\ambyte-IoT


## 1. The schedule

Edit the YAML here. Rules the device parser enforces:

- Block style only. `with: { protocol: MPF }` and `channels: [0]` are rejected; write them indented, one key per line.
- At most 8 steps per job, 16 jobs, 16 KiB in total.
- Every protocol a step names must be defined under `protocols:`.

To upload a file instead, replace the string with for example
`SCHEDULE_YAML = (REPO / "schedule" / "default.yaml").read_text(encoding="utf-8")`
(that line is also how you put the field default back).

In [2]:
SCHEDULE_YAML = """\
schema: jii.ambyte-schedule/v1-draft
id: urn:jii:schedule:mpf-staggered
version: 0.1.0-draft
name: mpf-staggered
description: MPF on AMBIT channels 0..3 one after the other, 5 s apart, every 10 minutes.

protocols:
  MPF:
    # Dark baseline.
    - pulses: 40
      freq: 10
      actinic: 0

    # Saturating phases. Actinic values are raw DAC values.
    - pulses: 70
      freq: 100
      actinic: -250
    - pulses: 10
      freq: 100
      actinic: -200
    - pulses: 10
      freq: 100
      actinic: -160
    - pulses: 10
      freq: 100
      actinic: -250

    # Relaxation.
    - pulses: 40
      freq: 10
      actinic: 0

jobs:
  mpf_staggered:
    schedule:
      cron: "*/10 * * * *"     # or: - dispatch   /   - boot
    steps:
      - uses: ambit/trace
        continue-on-error: true
        with:
          protocol: MPF
          channels:
            - 0
          hold_window: true
      - uses: device/sleep
        with:
          duration: 5s
      - uses: ambit/trace
        continue-on-error: true
        with:
          protocol: MPF
          channels:
            - 1
          hold_window: true
      - uses: device/sleep
        with:
          duration: 5s
      - uses: ambit/trace
        continue-on-error: true
        with:
          protocol: MPF
          channels:
            - 2
          hold_window: true
      - uses: device/sleep
        with:
          duration: 5s
      - uses: ambit/trace
        with:
          protocol: MPF
          channels:
            - 3
          hold_window: true
"""

# Labels the device stores and reports in `schedule release` and its STATUS heartbeat.
# No spaces allowed.
SCHEDULE_ID = "mpf-staggered-local"
SCRIPT_VERSION = "0.1.0-draft"
BUILT_AGAINST_FW = "local"

# The job cell 4 can fire on demand.
JOB_NAME = "mpf_staggered"

In [3]:
blob = SCHEDULE_YAML.replace("\r\n", "\n").encode("utf-8")
sha256 = hashlib.sha256(blob).hexdigest()

# Cheap local checks for the mistakes that are easy to make. The device compiler
# (step 3) is the real gate; this only fails sooner and with a line number.
problems = []
if len(blob) > 16 * 1024:
    problems.append(f"schedule is {len(blob)} bytes; the device accepts at most 16384")
for label in (SCHEDULE_ID, SCRIPT_VERSION, BUILT_AGAINST_FW):
    if not label or any(ch.isspace() for ch in label):
        problems.append(f"label {label!r} must be non-empty and contain no spaces")
for number, line in enumerate(SCHEDULE_YAML.splitlines(), start=1):
    if "\t" in line:
        problems.append(f"line {number}: tab character (indent with spaces)")
    if re.search(r"(:|^\s*-)\s+[\[{]", line.split("#", 1)[0]):
        problems.append(f"line {number}: flow-style {{...}} / [...] is not supported, "
                        "use indented block YAML")
if problems:
    raise ValueError("Schedule not uploadable:\n  " + "\n  ".join(problems))

print(f"schedule ok locally: {len(blob)} bytes, sha256 {sha256}")

schedule ok locally: 1543 bytes, sha256 98028d777f7a1087cbfe6367e43ba1a31cff15e8dddeaed023a7513bd65defd8


## 2. Find the Ambyte

Scans the Espressif USB-Serial-JTAG ports and keeps the ones that answer with the
`ambyte> ` prompt. With exactly one Ambyte attached nothing needs to be set. With
several attached, set `PORT` to the one you want.

In [4]:
PORT = None   # e.g. "COM7" or "/dev/ttyACM0" to skip auto-detection


def find_ambytes(port=None, prompt_timeout=5.0):
    """[(port, mac, device_name)] for every port with a live ambyte console."""
    candidates = [port] if port else ambyte_serial.esp_jtag_ports()
    if not candidates:
        print("No Espressif USB-Serial-JTAG port is enumerated.")
    found = []
    for candidate in candidates:
        probe = ambyte_serial.probe_device(candidate, timeout=prompt_timeout, retries=1)
        if not probe.is_ambyte:
            print(f"{candidate}: no ambyte prompt (still booting, held by another "
                  "program, or not an Ambyte)")
            continue
        name = probe.device_name or ""
        if probe.mac:
            name = ambyte_serial.expand_mac_token(name, probe.mac)
        print(f"{candidate}: Ambyte  MAC {probe.mac or '?'}  name {name or '?'}")
        found.append((candidate, probe.mac, name))
    return found


ambytes = find_ambytes(PORT)
if not ambytes:
    raise RuntimeError("No Ambyte found. Check the cable, close other serial monitors, "
                       "wait ~30 s after plugging in, then re-run this cell.")
if len(ambytes) > 1:
    raise RuntimeError("Several Ambytes answered; set PORT above to one of: "
                       + ", ".join(port for port, _, _ in ambytes))

TARGET_PORT, TARGET_MAC, TARGET_NAME = ambytes[0]
print(f"\ntarget: {TARGET_PORT}  {TARGET_MAC}  {TARGET_NAME}")

COM30: Ambyte  MAC 14:63:93:39:3F:A4  name AMBYTE_14:63:93:39:3F:A4

target: COM30  14:63:93:39:3F:A4  AMBYTE_14:63:93:39:3F:A4


## 3. Upload, validate, install

The schedule is staged on the device, compiled there with `schedule validate`, and only
then committed. A rejected schedule is discarded and the device keeps what it was running.

In [5]:
INSTALL_DEADLINE_S = 180


def body(reply):
    """A console reply without the echoed command and the trailing prompt."""
    lines = [line.rstrip() for line in reply.replace(PROMPT, "").splitlines()]
    return "\n".join(line for line in lines[1:] if line)


def open_console(port, mac):
    con = AmbyteConsole(port)
    try:
        if not con.wait_prompt(15.0):
            raise ConsoleError(f"{port}: no ambyte prompt; re-run the detect cell")
        seen = con.status_mac()
        if mac and seen != mac:
            raise ConsoleError(f"{port} now answers as {seen}, not {mac}; "
                               "re-run the detect cell")
    except Exception:
        con.close()
        raise
    return con


def upload_schedule(port, mac, blob, sha256, log=print):
    with open_console(port, mac) as con:
        release = stage_and_install(con, blob, sha256, log)
        log("")
        log(body(con.command("schedule status", timeout=10.0)))
        return release


def stage_and_install(con, blob, sha256, log):
    try:
        before = con.schedule_release()
        log(f"active before: version={before.script_version or '-'} "
            f"sha256={before.sha256[:12]}...")
        if before.sha256 == sha256 and before.running:
            log("this exact schedule is already installed and running; nothing to do")
            return before
    except ConsoleError:
        log("active before: no installed schedule identity (embedded default)")

    # Stage: the device keeps the bytes in /littlefs/schedule.yaml.new.
    reply = con.command("schedule begin", timeout=10.0)
    if "schedule begin: ready" not in reply:
        raise ConsoleError("this firmware has no serial schedule push "
                           "(needs v2.0.0 or later):\n" + body(reply))
    try:
        sent = 0
        for offset in range(0, len(blob), SCHEDULE_PUT_CHUNK_BYTES):
            chunk = blob[offset:offset + SCHEDULE_PUT_CHUNK_BYTES]
            reply = con.command(
                "schedule put " + base64.b64encode(chunk).decode("ascii"), timeout=15.0)
            sent += len(chunk)
            match = re.search(r"schedule put:\s*(\d+)\s*bytes", reply)
            if FAILURE_MARK in reply or match is None or int(match.group(1)) != sent:
                raise ConsoleError(f"push failed at byte {sent}:\n" + body(reply))
        log(f"staged {sent} bytes")

        # Compile on the device before anything is swapped.
        reply = con.command("schedule validate", timeout=20.0)
        if "schedule.yaml.new: valid" not in reply:
            match = re.search(r"schedule\.yaml\.new:\s*(.*)", reply)
            reason = match.group(1).strip() if match else body(reply)
            raise ConsoleError(f"the device compiler rejected the schedule: {reason}")
        log("device compiler: valid")

        reply = con.command(
            f"schedule commit {sha256} {SCHEDULE_ID} {SCRIPT_VERSION} {BUILT_AGAINST_FW}",
            timeout=15.0)
        if "schedule commit queued:" not in reply or FAILURE_MARK in reply:
            raise ConsoleError("commit refused:\n" + body(reply))
    except ConsoleError:
        try:
            con.command("schedule abort", timeout=10.0)   # drop the staged file
        except ConsoleError:
            pass
        raise

    # The firmware hashes, compiles, swaps and restarts the runner on its own
    # worker; `schedule release` reporting our digest is the confirmation.
    log("commit queued; waiting for the device to install it...")
    deadline = time.time() + INSTALL_DEADLINE_S
    last = "no answer yet"
    next_note = time.time() + 15.0
    while time.time() < deadline:
        time.sleep(2.0)
        try:
            now = con.schedule_release(timeout=20.0)
        except ConsoleError as exc:
            last = str(exc).splitlines()[0]
            if "went away" in last:
                raise
            continue
        if now.sha256 == sha256 and now.running:
            log(f"installed and running: version={now.script_version} "
                f"sha256={now.sha256} verified={now.verified}")
            return now
        last = f"device reports sha256={now.sha256[:12]}... running={now.running}"
        if time.time() >= next_note:
            next_note = time.time() + 15.0
            log(f"  still waiting ({last})")
    raise ConsoleError(f"not confirmed within {INSTALL_DEADLINE_S}s ({last}). "
                       "The device log has the reason; the previous schedule is "
                       "still on the device.")


release = upload_schedule(TARGET_PORT, TARGET_MAC, blob, sha256)

active before: no installed schedule identity (embedded default)
staged 1543 bytes
device compiler: valid
commit queued; waiting for the device to install it...
installed and running: version=0.1.0-draft sha256=98028d777f7a1087cbfe6367e43ba1a31cff15e8dddeaed023a7513bd65defd8 verified=True

schedule: RUNNING  source=INSTALLED
  sha256=98028d777f7a1087cbfe6367e43ba1a31cff15e8dddeaed023a7513bd65defd8
  id=urn:jii:schedule:mpf-staggered version=0.1.0-draft workbook=- name=mpf-staggered
  job              next due (UTC)         runs  fail  skip streak  last ms
  mpf_staggered    2026-11-10 08:30:00       0     0     0      0        0


## 4. Optional: fire the job now

Dispatches the job once instead of waiting for its cron slot, prints whatever the device
logs while it runs, then shows the job counters (`runs`, `fail`, `last ms`).
The staggered MPF takes about a minute with four AMBITs attached.

In [ ]:
WATCH_S = 90

with open_console(TARGET_PORT, TARGET_MAC) as con:
    print(body(con.command(f"schedule run {JOB_NAME}", timeout=10.0)))
    deadline = time.time() + WATCH_S
    while time.time() < deadline:
        text = ambyte_serial._ANSI_RE.sub("", con._read_available())   # raw device log
        if text:
            print(text, end="")
    print()
    print(body(con.command("schedule status", timeout=10.0)))